# 1.0 Verify AWS and Bedrock Access

Run this notebook before building the graph in `1.1_build_graph.ipynb`. It checks the exact AWS paths that Module 1 needs: temporary Vocareum credentials, the `us-east-1` region, Claude Sonnet 4.6, and Amazon Nova multimodal embeddings.

You do not enter AWS keys here. Vocareum injects temporary credentials into the notebook environment. This notebook does not create resources, change permissions, subscribe models, or repair account access.


## Prepare workshop dependencies

Run this cell first. It installs the pinned workshop packages once per lab
session when needed, then makes the shared dependency directory available to
this notebook.

In [ ]:
# Vocareum dependency bootstrap
import hashlib
import os
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

# Find requirements.txt from wherever this notebook was opened.
start = Path.cwd().resolve()
sources = (
    start / "notebooks" / "requirements.txt",
    start / "requirements.txt",
    start.parent / "requirements.txt",
)
requirements = next((path for path in sources if path.is_file()), None)
if requirements is None:
    looked = "\n  ".join(str(path) for path in sources)
    raise FileNotFoundError(
        f"No requirements.txt found from {start}. Looked at:\n  {looked}\n"
        "Open this notebook from the extracted starter-code tree rather than "
        "from a copy somewhere else in the workspace."
    )

# Pick an install directory every kernel in this session can read.
# Local disk first, because /voc/work is a slow network filesystem.
# WORKSHOP_DEPS_DIR overrides the search.
override = os.environ.get("WORKSHOP_DEPS_DIR")
if override:
    roots = (Path(override).expanduser().resolve(),)
else:
    roots = tuple(
        root / f"workshop-deps-{os.getuid()}"
        for root in (Path("/tmp"), Path.home(), requirements.parent)
    )
target = None
refusals = []
for candidate in roots:
    try:
        candidate.mkdir(mode=0o700, parents=True, exist_ok=True)
        probe = candidate / ".write-test"
        probe.write_text("")
        probe.unlink()
    except OSError as error:
        refusals.append(f"{candidate}: {error}")
        continue
    target = candidate
    break
if target is None:
    refused = "\n  ".join(refusals)
    if override:
        remedy = (
            f"WORKSHOP_DEPS_DIR names {override}, which this account cannot "
            "write. Unset it to fall back to the normal search."
        )
    else:
        remedy = (
            "Set WORKSHOP_DEPS_DIR to a directory you can write, then re-run "
            "this cell."
        )
    raise RuntimeError(
        f"No writable directory for the workshop dependencies.\n  {refused}\n"
        f"{remedy}"
    )

# Report the interpreter, the target filesystem, and free space, and warn
# when either would make the install slow or fail.
fstype = "unknown"
try:
    longest = ""
    for line in Path("/proc/mounts").read_text().splitlines():
        fields = line.split()
        if len(fields) < 3:
            continue
        point = fields[1].rstrip("/") + "/"
        if f"{target}/".startswith(point) and len(point) > len(longest):
            longest, fstype = point, fields[2]
except OSError:
    pass
free_gb = shutil.disk_usage(target).free / 1e9
print(f"Python       {sys.version.split()[0]} at {sys.executable}")
print(f"Target       {target} ({fstype}, {free_gb:.1f} GB free)")
print(f"Requirements {requirements}")
if fstype.startswith(("nfs", "cifs", "fuse")):
    print(
        f"WARNING: {target} is on a network filesystem ({fstype}). Writing "
        "roughly 15,000 small files there has taken over ten minutes. Set "
        "WORKSHOP_DEPS_DIR to a local directory to avoid it."
    )
if free_gb < 1.0:
    print(
        f"WARNING: only {free_gb:.1f} GB free, and the dependency tree needs "
        "about 0.5 GB."
    )

INSTALL_TIMEOUT_SECONDS = 900

# Install only when requirements.txt has changed since the last install.
stamp = target / "requirements.sha256"
digest = hashlib.sha256(requirements.read_bytes()).hexdigest()
if not stamp.is_file() or stamp.read_text().strip() != digest:
    print(f"Installing workshop dependencies into {target}")
    began = time.monotonic()
    install = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--progress-bar",
            "off",
            "--timeout",
            "20",
            "--retries",
            "2",
            "--upgrade",
            "--target",
            str(target),
            "-r",
            str(requirements),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    # Print pip's output as it arrives, and stop the install if it overruns
    # the timeout.
    watchdog = threading.Timer(INSTALL_TIMEOUT_SECONDS, install.kill)
    watchdog.start()
    try:
        for line in install.stdout:
            print(line, end="")
        code = install.wait()
    finally:
        watchdog.cancel()
    elapsed = time.monotonic() - began
    if code != 0:
        if elapsed >= INSTALL_TIMEOUT_SECONDS:
            raise TimeoutError(
                f"Installing into {target} passed {INSTALL_TIMEOUT_SECONDS} "
                f"seconds and was stopped. That target is on {fstype}; if that "
                "is a network filesystem, set WORKSHOP_DEPS_DIR to a local "
                "directory such as /tmp/workshop-deps and re-run this cell."
            )
        raise RuntimeError(
            f"pip install failed with exit status {code}. The pip output above "
            f"ends with the reason. A partial install is safe to discard: delete "
            f"{target} and re-run this cell."
        )
    stamp.write_text(digest)
    print(f"Installed in {elapsed:.0f} seconds")
if str(target) not in sys.path:
    sys.path.insert(0, str(target))
print(f"Workshop dependencies ready in {target}")

## Step 1: Confirm the AWS identity and region

The credential check stops instead of letting `boto3` fall back to another local profile. The identity and account number are safe to include in a support request. Secret values are never printed.


In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

import workshop

# Printed first, so a support request names the workshop revision before
# anything else in this notebook can fail.
print(f"workshop {workshop.__version__}")

# Module 1.0 reads the .env files a few lines below instead of letting
# start_module do it, because loading them is what this step teaches.
NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module(
    "01-build-graph", load_env=False
)

import boto3
from botocore.exceptions import BotoCoreError, ClientError

from verify_environment import check_bearer_token, load_environment_files

load_environment_files(NOTEBOOKS_ROOT, REPO_ROOT)
check_bearer_token()

REQUIRED_REGION = "us-east-1"
region = (
    os.getenv("AWS_REGION")
    or os.getenv("AWS_DEFAULT_REGION")
    or boto3.Session().region_name
    or REQUIRED_REGION
)
session = boto3.Session(region_name=region)
credentials = session.get_credentials()

if credentials is None:
    raise RuntimeError("No AWS credentials are available. Start the Vocareum lab.")

frozen = credentials.get_frozen_credentials()
missing = [
    name
    for name, value in (
        ("access key", frozen.access_key),
        ("secret key", frozen.secret_key),
        ("session token", frozen.token),
    )
    if not value
]
if missing:
    raise RuntimeError("Missing temporary credential fields: " + ", ".join(missing))

identity = session.client("sts", region_name=region).get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"Identity: {identity['Arn']}")
print(f"Region: {region}")

if region != REQUIRED_REGION:
    raise RuntimeError(
        f"This workshop requires {REQUIRED_REGION}; the session selected {region}."
    )

print("PASS: credentials, identity, and region are ready.")

## Step 2: Invoke the models Module 1 uses

This makes three small calls: Sonnet 4.6, streaming Sonnet 4.6, and one 1,024-dimension Nova embedding. A model appearing in the Bedrock catalog is not enough. Only a successful invocation proves the account can run the workshop.


In [ ]:
SONNET_MODEL_ID = "us.anthropic.claude-sonnet-4-6"
NOVA_MODEL_ID = "amazon.nova-2-multimodal-embeddings-v1:0"
EXPECTED_EMBEDDING_DIMENSIONS = 1024
MESSAGES = [
    {"role": "user", "content": [{"text": "Reply with one word: ready"}]}
]
INFERENCE_CONFIG = {"maxTokens": 32}
NOVA_REQUEST = {
    "taskType": "SINGLE_EMBEDDING",
    "singleEmbeddingParams": {
        "embeddingPurpose": "GENERIC_INDEX",
        "embeddingDimension": EXPECTED_EMBEDDING_DIMENSIONS,
        "text": {"truncationMode": "END", "value": "access check"},
    },
}


def failure_kind(error):
    code = error.response.get("Error", {}).get("Code", "Unknown")
    message = error.response.get("Error", {}).get("Message", "")
    lowered = message.lower()
    if "error 002" in lowered or "not allowed for this account" in lowered:
        return "ACCOUNT ENTITLEMENT", code, message
    if code in {"AccessDenied", "AccessDeniedException", "UnauthorizedOperation"}:
        return "IAM OR ORGANIZATION POLICY", code, message
    return "AWS SERVICE ERROR", code, message


def print_response_metadata(error):
    """Print the HTTP status, the request id, and every header AWS returned."""
    metadata = error.response.get("ResponseMetadata")
    if not metadata:
        # Not every exception carries ResponseMetadata. A call that failed
        # before AWS answered has no reply to describe.
        print("  No ResponseMetadata, so no HTTP reply reached this kernel.")
        return
    print(f"  HTTP status: {metadata.get('HTTPStatusCode', 'unknown')}")
    print(f"  AWS RequestId: {metadata.get('RequestId', 'none')}")
    headers = metadata.get("HTTPHeaders") or {}
    if not headers:
        print("  Response headers: none")
        return
    print("  Response headers:")
    for name in sorted(headers):
        print(f"    {name}: {headers[name]}")


def check(label, operation):
    try:
        detail = operation()
    except ClientError as error:
        kind, code, message = failure_kind(error)
        print(f"FAIL: {label}")
        print(f"  {kind}: {code}: {message}")
        print_response_metadata(error)
        return False
    except (BotoCoreError, KeyError, TypeError, ValueError) as error:
        print(f"FAIL: {label}")
        print(f"  CLIENT OR RESPONSE ERROR: {error}")
        return False
    print(f"PASS: {label} - {detail}")
    return True


runtime = session.client("bedrock-runtime", region_name=region)


In [ ]:
import json


def invoke_sonnet():
    response = runtime.converse(
        modelId=SONNET_MODEL_ID,
        messages=MESSAGES,
        inferenceConfig=INFERENCE_CONFIG,
    )
    blocks = response["output"]["message"]["content"]
    text = "".join(block.get("text", "") for block in blocks).strip()
    if not text:
        raise ValueError("Sonnet returned no text")
    return f"answered {text[:40]!r}"


def stream_sonnet():
    response = runtime.converse_stream(
        modelId=SONNET_MODEL_ID,
        messages=MESSAGES,
        inferenceConfig=INFERENCE_CONFIG,
    )
    pieces = []
    for event in response["stream"]:
        delta = event.get("contentBlockDelta", {}).get("delta", {})
        if delta.get("text"):
            pieces.append(delta["text"])
    text = "".join(pieces).strip()
    if not text:
        raise ValueError("streaming Sonnet returned no text")
    return f"answered {text[:40]!r}"


def invoke_nova():
    response = runtime.invoke_model(
        modelId=NOVA_MODEL_ID,
        body=json.dumps(NOVA_REQUEST),
        contentType="application/json",
        accept="application/json",
    )
    payload = json.loads(response["body"].read())
    embeddings = payload.get("embeddings", [])
    vector = embeddings[0].get("embedding", []) if embeddings else []
    if len(vector) != EXPECTED_EMBEDDING_DIMENSIONS:
        raise ValueError(
            f"Nova returned {len(vector)} dimensions; "
            f"expected {EXPECTED_EMBEDDING_DIMENSIONS}"
        )
    return f"returned a {len(vector)}-dimension vector"


results = [
    check("Sonnet 4.6 InvokeModel", invoke_sonnet),
    check("Sonnet 4.6 streaming", stream_sonnet),
    check("Nova multimodal embeddings", invoke_nova),
]

if not all(results):
    raise RuntimeError(
        "Bedrock access is not ready. Copy the account, region, and failing line "
        "into the support request. Do not continue to 1.1."
    )

print("\nEnvironment is ready. Continue to 1.1_build_graph.ipynb.")


## Reading a failure

| Result | Meaning | Owner |
|---|---|---|
| `ACCOUNT ENTITLEMENT` and Error 002 | AWS blocks model use for this allocated account | Vocareum or AWS account-pool administrator |
| `IAM OR ORGANIZATION POLICY` | The session role or an organization policy denied the API action | Workshop template owner or Vocareum |
| Wrong region | The notebook is not using `us-east-1`, where the workshop models run | Lab configuration |
| Empty `AWS_BEARER_TOKEN_BEDROCK` | The CONFIG.txt line is uncommented with no key after it, so botocore sends an empty bearer header and never falls back | Whoever edited CONFIG.txt |
| `_ARRAY_API not found` and `compiled using NumPy 1.x` | The lab image ships a NumPy 1.x build of `pyarrow`, which the Neo4j driver imports best-effort. The block appears in `1.1` or a later notebook, and the driver discards the failed import, so that notebook keeps running | Workshop requirements, where `pyarrow>=16.1.0` replaces that build |
| All checks pass | The account can begin Module 1 | Continue to `1.1_build_graph.ipynb` |

Every failing check now prints its HTTP status, its AWS RequestId, and every response header. Paste that whole block into the support request, because the RequestId is what AWS support looks up.

Seeing a model in the Bedrock catalog does not prove it can be invoked. Error 002 is not repaired by changing this notebook, adding AWS keys, or adding another IAM allow statement to the workshop template.
